# Spoeqi Quine Pact Pipeline

End-to-end demonstration of the spoeqi class-4 quine pipeline:

1. Generate a K=4 hex CA rule (16,384 bytes = 128×128 LUT)
2. Run the rule on its own LUT-as-image — the metachain step
3. Measure self-reproduction score (strict + arbitrary-σ)
4. Classify the rule (Wolfram class + continuous c4 score)
5. Walk the chain, find class-4 streaks, detect fixed points
6. Pack output streams (4 cells/byte) and unpack to cell view

**Setup:** must be run from inside the Velour project with the venv (`venv/bin/jupyter`).

In [ ]:
import os, sys, django
REPO = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, REPO)
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'velour.settings')
django.setup()
import numpy as np
from spoeqi import metachain as mc
from spoeqi import keychain as kc
print('repo:', REPO)

## 1. Generate a random K=4 rule
Each rule is exactly 16,384 bytes (one byte per LUT entry, only the low 2 bits used).

In [ ]:
rng = np.random.default_rng(0xC4FE)
rule = bytes(rng.integers(0, 4, size=16384, dtype=np.uint8))
print(f'rule size: {len(rule)} bytes')
print(f'first 16 entries: {list(rule[:16])}')
print(f'reshapes to 128x128: {len(rule) == 128*128}')

## 2-3. Metachain step + self-reproduction

The metachain step applies CA^16 to the LUT-as-image. A class-4 fixed point is a rule R with `CA^16(R.LUT) == R.LUT` AND `cls(R) == 4`.

In [ ]:
sr_strict = mc.self_reproduce_score(rule, ticks=16)
sr_arbs   = mc.sr_arbitrary_sigma(rule, ticks=16)
cls, c4   = mc.classify_rule(rule, probe_ticks=24)
act       = mc.probe_activity(rule, ticks=20)
print(f'sr_strict  (positional)   = {sr_strict:.4f}   (1.0 = exact fixed point)')
print(f'sr_arbsigma (histogram)   = {sr_arbs:.4f}   (more permissive)')
print(f'wolfram class             = {cls}            (4 = edge of chaos)')
print(f'continuous c4_score       = {c4:.4f}')
print(f'probe activity            = {act:.4f}')

## 4. Walk the metachain
Detect class-4 streaks and fixed points up to N levels.

In [ ]:
walk = mc.walk_chain(rule, depth=24)
print(f'class4_run_length (start-prefix): {walk["class4_run_length"]}')
for lv in walk['levels'][:8]:
    print(f'  L{lv["level"]:>2}  cls={lv["class"]} c4={lv["c4"]:.3f} act={lv["act"]:.3f} '
          f'sr_arbsigma={lv["sr_arbsigma"]:.3f}')

## 5. Saved quines: load #122 (the breakthrough)
Quine #122 is a 13-byte mutation of #116 whose chain converges at L130 to a self-mapping class-4 rule — effective chain depth = ∞.

In [ ]:
from caformer.models import ComponentChampion
q122 = ComponentChampion.objects.get(pk=122)
print(f'#122: {q122.run_label}  fitness={q122.fitness:.4f}')
q122_rule = bytes(q122.rules_blob)
print(f'sr_strict(#122) = {mc.self_reproduce_score(q122_rule, ticks=16):.4f}')
print(f'sr_arbsigma(#122) = {mc.sr_arbitrary_sigma(q122_rule, ticks=16):.4f}')

## 6. Pact: pack stream + unpack to cell view
Same on-disk bytes, two read interpretations: packed (4 cells/byte = file content) and cells (1 byte per cell, 0..3 = UI masks).

In [ ]:
stream_cells = mc.run_ca_stream(rule, init_seed=0x1234, ticks=16, packed=False)
stream_packed = mc.pack_k4_stream(stream_cells)
stream_unpacked = mc.unpack_k4_bytes(stream_packed)
print(f'raw cells:      {len(stream_cells):>6} bytes (1 byte/cell)')
print(f'packed:         {len(stream_packed):>6} bytes (4 cells/byte)')
print(f'unpacked again: {len(stream_unpacked):>6} bytes')
print(f'round-trip exact: {stream_cells == stream_unpacked}')

## 7. Keychain backend: time-evolving + tag a region
The keychain DB regenerates from a seed; tags can address packed bytes (file content) or cells (UI mask) — same backend, two views.

In [ ]:
backend = kc.ChainBackend(q122_rule, kc.ChainParams(depth=3, packed=True))
packed = backend.read(level=0, stream_index=0, start=0, end=16)
cells  = backend.read_cells(level=0, stream_index=0, cell_start=0, cell_end=64)
print('packed[0:16]:', list(packed))
print('cells[0:64]: ', list(cells))
print('cell view == unpack(packed)?', bytes(cells) == mc.unpack_k4_bytes(packed))

## Where to go next

- **Deeper chains:** `manage.py deep_chain_search` and the saved class-4 quines list
- **Visual exploration:** `/ouroboros/` web app — every saved quine + live self-application demo
- **65K vocab via ALICE:** `manage.py alice_bundle_qrpair_vocab` (self-contained Slurm bundle)
- **CA chat:** `/caformer/chat/` — per-prompt dispatcher routes to trained QRPair models